# Old Photo Restore - Dataset Preprocessing and Mask Generation

This notebook preprocesses `joshuachin/openphoto-restore-dataset` into a dataset suitable for **mask-based image inpainting** and LoRA fine-tuning.

The original dataset provides paired `pristine_image` and `damaged_image` samples but does not include inpainting masks. This notebook derives local damage masks from the image pairs, checks mask quality, filters unsuitable samples, and exports a train/test dataset with `damaged`, `pristine`, and `masks` folders.

The processed dataset is then packaged as a Kaggle Dataset so it can be reused by the LoRA training and evaluation notebooks.

## Kaggle Setup

Recommended Kaggle settings:

- Accelerator: GPU is not required for this notebook.
- Internet: ON, because the dataset is downloaded from Hugging Face.
- Runtime: CPU is fine.

If `datasets` is already installed, the install cell will finish quickly.

In [ ]:
!pip -q install datasets

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import load_dataset
from PIL import Image, ImageFilter, ImageOps
from tqdm.auto import tqdm

DATASET_ID = "joshuachin/openphoto-restore-dataset"
IMAGE_SIZE = 512
INSPECT_SAMPLES = 100
TRAIN_EXPORT_LIMIT = 3000
TEST_EXPORT_LIMIT = 500
KAGGLE_USERNAME = "dwctien"
KAGGLE_DATASET_SLUG = "openphoto-restore-filtered-inpainting-subset"

OUTPUT_ROOT = Path("/kaggle/working/openphoto_mask_check")
PREVIEW_DIR = OUTPUT_ROOT / "previews"
EXPORT_DIR = OUTPUT_ROOT / "inpainting_dataset"
TRAIN_EXPORT_DIR = EXPORT_DIR / "train"
TEST_EXPORT_DIR = EXPORT_DIR / "test"

PREVIEW_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
TEST_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Outputs will be saved to: {OUTPUT_ROOT}")

## Load Dataset

The dataset has paired damaged and pristine images. We use a small slice first because this notebook is for feasibility checking.

In [ ]:
train_ds = load_dataset(DATASET_ID, split=f"train[:{INSPECT_SAMPLES}]")
test_ds = load_dataset(DATASET_ID, split="test[:50]")

print(train_ds)
print(train_ds.column_names)
print(test_ds)

## Mask Derivation

A direct RGB difference often mistakes global sepia/fade/grayscale effects for local damage. This function uses grayscale residuals, optional autocontrast normalization, thresholding, dilation, and simple area filtering.

Mask convention: white pixels are regions to inpaint.

In [ ]:
def prepare_image(image, size=IMAGE_SIZE):
    return image.convert("RGB").resize((size, size), Image.LANCZOS)


def derive_damage_mask(
    pristine,
    damaged,
    size=IMAGE_SIZE,
    threshold=35,
    blur_radius=1.0,
    dilate_size=5,
    autocontrast=True,
):
    pristine = prepare_image(pristine, size)
    damaged = prepare_image(damaged, size)

    pristine_gray = ImageOps.grayscale(pristine)
    damaged_gray = ImageOps.grayscale(damaged)

    if autocontrast:
        pristine_gray = ImageOps.autocontrast(pristine_gray)
        damaged_gray = ImageOps.autocontrast(damaged_gray)

    p = np.asarray(pristine_gray).astype(np.int16)
    d = np.asarray(damaged_gray).astype(np.int16)
    diff = np.abs(p - d).astype(np.uint8)

    diff_img = Image.fromarray(diff, mode="L")
    if blur_radius > 0:
        diff_img = diff_img.filter(ImageFilter.GaussianBlur(blur_radius))

    diff_arr = np.asarray(diff_img)
    mask_arr = (diff_arr > threshold).astype(np.uint8) * 255
    mask = Image.fromarray(mask_arr, mode="L")

    if dilate_size and dilate_size > 1:
        if dilate_size % 2 == 0:
            dilate_size += 1
        mask = mask.filter(ImageFilter.MaxFilter(dilate_size))

    mask_area = float(np.asarray(mask).mean() / 255.0)
    return mask, mask_area, diff_img


def make_masked_preview(damaged, mask, fill=(0, 0, 0)):
    damaged = prepare_image(damaged)
    mask_arr = np.asarray(mask.convert("L"))
    image_arr = np.asarray(damaged).copy()
    image_arr[mask_arr > 127] = fill
    return Image.fromarray(image_arr, mode="RGB")


def is_good_mask(mask_area, min_area=0.005, max_area=0.40):
    return min_area <= mask_area <= max_area

## Visual Check

Inspect these grids manually. Good masks should focus on local scratches/cracks/dust/tears. Bad masks usually cover most of the image because of global aging effects.

In [ ]:
def show_sample_grid(dataset, indices, threshold=35, max_cols=4):
    rows = len(indices)
    fig, axes = plt.subplots(rows, max_cols, figsize=(4 * max_cols, 4 * rows))
    if rows == 1:
        axes = np.expand_dims(axes, axis=0)

    for row, idx in enumerate(indices):
        sample = dataset[idx]
        pristine = sample["pristine_image"]
        damaged = sample["damaged_image"]
        mask, mask_area, diff_img = derive_damage_mask(pristine, damaged, threshold=threshold)
        masked = make_masked_preview(damaged, mask)

        images = [
            (prepare_image(damaged), "Damaged"),
            (prepare_image(pristine), "Pristine"),
            (diff_img, "Residual"),
            (mask, f"Mask {mask_area:.1%}"),
        ]

        for col, (img, title) in enumerate(images):
            ax = axes[row, col]
            ax.imshow(img, cmap="gray" if img.mode == "L" else None)
            ax.set_title(f"#{idx} - {title}")
            ax.axis("off")

    plt.tight_layout()
    plt.show()


show_sample_grid(train_ds, list(range(8)), threshold=35)

## Mask Area Statistics

This summarizes how many samples pass the simple area filter. If too many masks are huge, the dataset is noisy for pure inpainting unless you improve mask generation or filter more aggressively.

In [ ]:
records = []

for idx, sample in enumerate(tqdm(train_ds, desc="Deriving masks")):
    mask, mask_area, _ = derive_damage_mask(
        sample["pristine_image"],
        sample["damaged_image"],
        threshold=35,
    )
    records.append({
        "index": idx,
        "mask_area": mask_area,
        "keep": is_good_mask(mask_area),
    })

stats_df = pd.DataFrame(records)
display(stats_df.describe())
display(stats_df["keep"].value_counts().rename("count"))

plt.figure(figsize=(8, 4))
plt.hist(stats_df["mask_area"], bins=30)
plt.axvline(0.005, color="green", linestyle="--", label="min keep")
plt.axvline(0.40, color="red", linestyle="--", label="max keep")
plt.xlabel("Mask area ratio")
plt.ylabel("Sample count")
plt.legend()
plt.show()

## Inspect Kept And Rejected Samples

You want kept samples to look like local inpainting problems. Rejected large-mask samples are useful evidence for the report: OpenPhoto contains global aging/color effects that are not pure inpainting.

In [ ]:
kept_indices = stats_df.loc[stats_df["keep"], "index"].head(8).tolist()
large_reject_indices = stats_df.loc[stats_df["mask_area"] > 0.40, "index"].head(8).tolist()

print("Kept indices:", kept_indices)
if kept_indices:
    show_sample_grid(train_ds, kept_indices, threshold=35)

print("Large rejected indices:", large_reject_indices)
if large_reject_indices:
    show_sample_grid(train_ds, large_reject_indices, threshold=35)

## Export A Complete Inpainting Dataset

This creates a local dataset with separate `train` and `test` folders. Use it only if the visual check above looks reasonable.

Folder format:

```text
/kaggle/working/openphoto_mask_check/inpainting_dataset/
  train/
    damaged/000000.png
    pristine/000000.png
    masks/000000.png
    metadata.csv
  test/
    damaged/000000.png
    pristine/000000.png
    masks/000000.png
    metadata.csv
  metadata_all.csv
```

In [ ]:
def export_inpainting_split(dataset, split_name, export_dir, limit, threshold=35):
    damaged_dir = export_dir / "damaged"
    pristine_dir = export_dir / "pristine"
    masks_dir = export_dir / "masks"

    for folder in [damaged_dir, pristine_dir, masks_dir]:
        folder.mkdir(parents=True, exist_ok=True)

    rows = []
    export_id = 0

    for source_idx, sample in enumerate(tqdm(dataset, desc="Exporting subset")):
        pristine = prepare_image(sample["pristine_image"])
        damaged = prepare_image(sample["damaged_image"])
        mask, mask_area, _ = derive_damage_mask(pristine, damaged, threshold=threshold)

        if not is_good_mask(mask_area):
            continue

        name = f"{export_id:06d}.png"
        damaged.save(damaged_dir / name)
        pristine.save(pristine_dir / name)
        mask.save(masks_dir / name)

        rows.append({
            "split": split_name,
            "id": export_id,
            "source_index": source_idx,
            "damaged_path": f"{split_name}/damaged/{name}",
            "pristine_path": f"{split_name}/pristine/{name}",
            "mask_path": f"{split_name}/masks/{name}",
            "mask_area": mask_area,
            "prompt": "restore an old damaged photo",
        })

        export_id += 1
        if export_id >= limit:
            break

    metadata = pd.DataFrame(rows)
    metadata.to_csv(export_dir / "metadata.csv", index=False)
    return metadata


full_train_ds = load_dataset(DATASET_ID, split="train")
full_test_ds = load_dataset(DATASET_ID, split="test")

train_metadata = export_inpainting_split(
    full_train_ds,
    split_name="train",
    export_dir=TRAIN_EXPORT_DIR,
    limit=TRAIN_EXPORT_LIMIT,
    threshold=35,
)
test_metadata = export_inpainting_split(
    full_test_ds,
    split_name="test",
    export_dir=TEST_EXPORT_DIR,
    limit=TEST_EXPORT_LIMIT,
    threshold=35,
)

metadata = pd.concat([train_metadata, test_metadata], ignore_index=True)
metadata.to_csv(EXPORT_DIR / "metadata_all.csv", index=False)

print(f"Exported {len(train_metadata)} train samples to {TRAIN_EXPORT_DIR}")
print(f"Exported {len(test_metadata)} test samples to {TEST_EXPORT_DIR}")
print(f"Complete dataset root: {EXPORT_DIR}")
display(metadata.head())

## Package As Kaggle Dataset

Run this after exporting the filtered subset. It creates a Kaggle-ready dataset folder and a `.zip` archive in `/kaggle/working`.

Recommended flow:

1. Run the export cell above.
2. Run the packaging cells below.
3. Run the Kaggle CLI publish cell if your Kaggle credentials are available.
4. Add the published dataset as an input to the LoRA training notebook.

In [ ]:
import json
import shutil

KAGGLE_DATASET_DIR = Path("/kaggle/working/openphoto_inpainting_lora_dataset")
KAGGLE_DATASET_ZIP = Path("/kaggle/working/openphoto_inpainting_lora_dataset.zip")

if KAGGLE_DATASET_DIR.exists():
    shutil.rmtree(KAGGLE_DATASET_DIR)

shutil.copytree(EXPORT_DIR, KAGGLE_DATASET_DIR)

dataset_metadata = {
    "title": "OpenPhoto Restore Filtered Inpainting Subset",
    "id": f"{KAGGLE_USERNAME}/{KAGGLE_DATASET_SLUG}",
    "licenses": [{"name": "CC-BY-4.0"}],
}

with open(KAGGLE_DATASET_DIR / "dataset-metadata.json", "w", encoding="utf-8") as f:
    json.dump(dataset_metadata, f, indent=2)

readme_text = f"""# OpenPhoto Restore Filtered Inpainting Subset

This dataset was derived from `joshuachin/openphoto-restore-dataset` for an image inpainting LoRA experiment.

The dataset contains separate `train` and `test` splits. Each split contains:

- `damaged/*.png`: old/damaged input image
- `pristine/*.png`: clean target image
- `masks/*.png`: derived binary inpainting mask, where white pixels are regions to restore
- `metadata.csv`: sample paths, source index, mask area, and prompt

The root folder also contains `metadata_all.csv`.

Mask generation summary:

- resize to {IMAGE_SIZE}x{IMAGE_SIZE}
- grayscale residual between pristine and damaged image
- autocontrast normalization
- threshold = 35
- dilation = 5
- keep samples with 0.5% <= mask area <= 40%

Source dataset license: CC BY 4.0.
"""

with open(KAGGLE_DATASET_DIR / "README.md", "w", encoding="utf-8") as f:
    f.write(readme_text)

print(f"Kaggle-ready dataset folder: {KAGGLE_DATASET_DIR}")
print(f"Kaggle dataset id: {KAGGLE_USERNAME}/{KAGGLE_DATASET_SLUG}")

In [ ]:
if KAGGLE_DATASET_ZIP.exists():
    KAGGLE_DATASET_ZIP.unlink()

shutil.make_archive(
    base_name=str(KAGGLE_DATASET_ZIP.with_suffix("")),
    format="zip",
    root_dir=KAGGLE_DATASET_DIR,
)

file_count = sum(1 for path in KAGGLE_DATASET_DIR.rglob("*") if path.is_file())
zip_size_mb = KAGGLE_DATASET_ZIP.stat().st_size / (1024 * 1024)

print(f"Dataset files: {file_count}")
print(f"Zip archive: {KAGGLE_DATASET_ZIP}")
print(f"Zip size: {zip_size_mb:.1f} MB")

## Publish To Kaggle Dataset

This cell publishes the prepared folder as a Kaggle Dataset under `dwctien/openphoto-restore-filtered-inpainting-subset`.

If it fails with an authentication error, add your Kaggle API token to the notebook secrets or upload the generated `.zip` manually through the Kaggle Dataset UI.

In [ ]:
import subprocess

dataset_id = f"{KAGGLE_USERNAME}/{KAGGLE_DATASET_SLUG}"

def run_kaggle_command(args):
    result = subprocess.run(args, capture_output=True, text=True)
    print("$", " ".join(args))
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    return result


publish = run_kaggle_command([
    "kaggle",
    "datasets",
    "create",
    "-p",
    str(KAGGLE_DATASET_DIR),
    "--dir-mode",
    "zip",
])

if publish.returncode != 0 and "already" in (publish.stdout + publish.stderr).lower():
    publish = run_kaggle_command([
        "kaggle",
        "datasets",
        "version",
        "-p",
        str(KAGGLE_DATASET_DIR),
        "-m",
        "Update filtered OpenPhoto inpainting train/test dataset",
        "--dir-mode",
        "zip",
    ])

if publish.returncode == 0:
    print(f"Published dataset: https://www.kaggle.com/datasets/{dataset_id}")
else:
    print("Publish failed. You can still upload the generated zip manually:")
    print(KAGGLE_DATASET_ZIP)

## Output

This notebook produces a Kaggle-ready dataset for old photo inpainting. The output dataset contains separate `train` and `test` splits, and each sample includes a damaged input image, a pristine target image, and a derived binary mask where white pixels indicate regions to inpaint.

The generated masks are derived automatically from paired damaged-pristine images rather than manually annotated. Therefore, residual-based mask noise remains a limitation of the processed dataset, especially for samples with global color, contrast, or aging effects.

The final packaged dataset is intended to be used as input for the LoRA fine-tuning and evaluation notebooks.